# Analysis — CLIP zero-shot

**Owner:** Collaborator A   |   **Model dir:** `results/clip/`

What this notebook adds: the numbers behind the verdict. It loads the saved
metrics + predictions produced by the eval script, redraws the confusion
matrix for this model, and documents the *manual error inspection* — the
core of the project's analysis strategy.


## 1. Headline metrics

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

results = Path("results") / "clip"
metrics = json.loads((results / "metrics.json").read_text()) if (results / "metrics.json").exists() else None
y_true = np.load(results / "y_true.npy") if (results / "y_true.npy").exists() else None
y_pred = np.load(results / "y_pred.npy") if (results / "y_pred.npy").exists() else None

if metrics:
    print(f"accuracy: {metrics['accuracy']:.4f}")
    import pandas as pd
    print(pd.DataFrame(metrics["classification_report"]).T.round(3))
else:
    print("No metrics yet — run `fauxtography eval-clip` (for CLIP) first.")


## 2. Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix
from fauxtography.data.dataset import CLASS_NAMES, REAL_IDX, FAKE_IDX

if y_true is not None:
    cm = confusion_matrix(y_true, y_pred, labels=[REAL_IDX, FAKE_IDX])
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(CLASS_NAMES); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center")
    plt.title("Confusion matrix")
    plt.show()
    # Where do we fail? (0,1)=real mislabelled fake; (1,0)=fake passed as real
else:
    print("No predictions yet (see cell 1).")


## 3. Manual error inspection

In [ ]:
# Load a few misclassified images and inspect what confused the model.
# Fill in: what visual artifacts / styles fooled the model? (presentation gold.)
# NOTE: to see the actual images, load the test set with fauxtography
# (FauxtoDataset) and reconstruct the misclassified file paths via the
# dataset's .samples attribute. Fill in your observations below.
import numpy as np
if y_true is not None:
    errors = np.where(y_true != y_pred)[0]
    print(f"{len(errors)} misclassified (of {len(y_true)}): {len(errors)/len(y_true):.1%}")
    for idx in errors[:6]:
        print("sample index", idx, "true=", CLASS_NAMES[y_true[idx]],
              "pred=", CLASS_NAMES[y_pred[idx]])
else:
    print("Nothing to inspect yet.")


## 4. Verdict

In [ ]:
# Write your verdict here (this is what goes on the slides):
#   - Where does this model stand versus the others?
#   - What kind of image fools it?
#   - One-sentence takeaway.
print("TODO: verdict for this model — see docs/analysis_<model>.md for the write-up")
